# Tutorial: Application Gateway Module — Issue-Driven Development

**Module**: `Azure/terraform-azurerm-avm-res-network-applicationgateway`  
**Scenario**: Working through an upstream issue (e.g., migrating a resource from azurerm to AzAPI)
using the avm-contributor-agent maker/checker pipeline.

This tutorial is a walkthrough of the **issue-driven** workflow, showing what each stage
produces. For an upgrade-specific tutorial (dispatch `upgrade-tests.yml` to evidence
base → head compatibility), see `aca-managed-environment-dev.ipynb`.

## 1. Prerequisites

```bash
# Authenticate GitHub CLI (once per machine)
gh auth login
gh auth status   # verify

# Set your dispatch token in .env
# AGENT_DISPATCH_TOKEN=github_pat_...
# (fine-grained PAT: kewalaka/avm-contributions, Actions RW + Contents R + Metadata R)

# Pre-commit the module fork to generate its AVM skill
cd ~/path/to/your/fork
./avm pre-commit
# writes: .agents/skills/AVM-Terraform-Development/SKILL.md
```

In [ ]:
import os, json, subprocess, sys
from pathlib import Path

# Load .env
from dotenv import load_dotenv
load_dotenv(override=True)

# Verify config
from config import config
print(f"Endpoint:       {(config.project_endpoint or '(not set)')[:50]}")
print(f"Dispatch token: {'set' if os.environ.get('AGENT_DISPATCH_TOKEN') else 'NOT SET'}")

r = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
print(f"gh auth:        {'ok' if r.returncode == 0 else 'NOT LOGGED IN'}")

## 2. Run the dev pipeline

The primary entry point is `python main.py dev`. Replace the issue number with a real
open issue on the upstream module.

```bash
python main.py dev \
  --upstream-repo Azure/terraform-azurerm-avm-res-network-applicationgateway \
  --issue 42 \
  --fork-owner YOUR_GITHUB_USERNAME
```

The pipeline stages:

```
Issue fetch
  └── fork sync  (ensure fork exists; fast-forward default branch)
        └── clone workspace  (~/.tfdev/ws/<run-id>/repo)
              └── Developer agent  (reads module AVM skill + additive instructions)
                    └── Reviewer agent  (gates diff; up to 3 iterations)
                          └── dispatch module-checks CI  → kewalaka/avm-contributions
                                └── dispatch module-e2e CI  → kewalaka/avm-contributions
                                      └── open draft PR on upstream
```

In [ ]:
# Show what a DevRequest looks like
sys.path.insert(0, str(Path.cwd().parent))
from request import DevRequest

req = DevRequest(
    upstream_repo="Azure/terraform-azurerm-avm-res-network-applicationgateway",
    issue_number=42,
    fork_owner="your-github-username",
)
print(req.summary())
print(f"Auto branch name: {req.auto_branch_name('azapi-migration')}")
print(f"Mode:             {req.mode}")

## 3. What the Developer agent does

The Developer loads two sets of instructions:

1. **Module AVM skill** — `.agents/skills/AVM-Terraform-Development/SKILL.md` from the
   cloned fork workspace. Generated by `./avm pre-commit`. Contains module-specific
   patterns, variable conventions, example structure.

2. **Additive instructions** — `agents/prompts/developer-additive.md`. Contains the
   AzAPI mandate, scope boundaries, commit format rules.

Key constraints the Developer enforces:
- All Azure resources use `azapi_resource` (not `azurerm_*` except `azuread_*` for Entra)
- AVM utility interfaces (`terraform-azure-avm-utl-interfaces`) used for all patterns
  except diagnostics (pending `feat/prepv1`)
- Commits include `Agent-Run-Id:` and `Co-authored-by: Copilot` trailers
- Branch prefix: `agent/issue-<N>-<slug>-<run-id-short>`

## 4. What the Reviewer agent checks

The Reviewer loads:
1. **AVM review skill** — `agents/skills/avm-review-skill.md` (forked from HashiCorp
   SKILL.md; ~37 requirements; AzAPI-first amendment applied)
2. **Additive review instructions** — `agents/prompts/reviewer-additive.md`

The Reviewer catches **all** issues, combining duplicate findings with instance counts.

Blocking findings (will reject the diff):
- `azurerm_*` usage outside the allowed exceptions
- Missing AVM utility interface where one exists
- Breaking change without `UPGRADE.md` update (TFNFR35)
- Non-conventional commit messages
- Force-push or branch prefix violation

If the Reviewer rejects 3 times on the same diff, the pipeline escalates:
opens a draft PR with the partial diff + a comment on the upstream issue
explaining why the agent stopped.

## 5. CI dispatch

On a Reviewer-approved diff, two CI jobs are dispatched to `kewalaka/avm-contributions`
via `repository_dispatch`:

| Event | Workflow | What it runs | Timeout |
|-------|----------|-------------|------|
| `module-checks` | `checks.yml` | pre-commit hooks, linting | ~10 min |
| `module-e2e` | `e2e-tests.yml` | `terraform apply` + idempotency per example | ~60 min |

The agent polls via `gh run watch` and downloads artifacts when each run completes.

In [ ]:
# Inspect a CI result from a previous run
# (replace with your actual run-id)
RUN_ID     = "REPLACE_WITH_RUN_ID"
checks_dir = Path.home() / ".tfdev" / "ws" / RUN_ID / "ci" / "checks"
e2e_dir    = Path.home() / ".tfdev" / "ws" / RUN_ID / "ci" / "e2e"

for label, path in [("checks", checks_dir / "summary.json"), ("e2e", e2e_dir / "summary.json")]:
    if path.exists():
        with open(path) as f:
            print(f"--- {label} ---")
            print(json.dumps(json.load(f), indent=2))
    else:
        print(f"{label}: not found at {path}")

## 6. The upstream PR

After CI passes, the agent opens (or updates) a **draft PR** on the upstream module repo.

The PR body uses managed regions that the agent regenerates on each push:

```markdown
## Summary
<!-- agent:summary -->
Brief description of what changed and why (from issue context).
<!-- /agent:summary -->

## CI Evidence
<!-- agent:evidence -->
- module-checks: ✅ https://github.com/kewalaka/avm-contributions/actions/runs/...
- module-e2e:    ✅ https://github.com/kewalaka/avm-contributions/actions/runs/...
  - default: apply success, idempotency clean
<!-- /agent:evidence -->
```

Everything outside those markers is human territory — safe to edit without being overwritten.

Flip to ready when satisfied:
```bash
gh pr ready <pr-number> -R Azure/terraform-azurerm-avm-res-network-applicationgateway
```